# Can PCA reduce the dimensionality of mobile feature data while preserving important information?
**How to use:** Run each cell in order with **Shift + Enter**. Read the explanation above each cell, then look at the output below it.
Dataset: Mobile Price Classification (`train.csv`, `test.csv`).

## Step 0 — Import libraries
These are the tools we need: pandas (tables), numpy (math), matplotlib/seaborn (plots), scikit-learn (PCA and models).

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, silhouette_score, mean_squared_error)
sns.set_theme(style="whitegrid")
RS = 42   # random seed so your results match

# A. Data Loading
## A1 — Upload your files
Click **Choose files** and select both `train.csv` and `test.csv`. They are saved in Colab's `/content/` folder.

In [ ]:
from google.colab import files
uploaded = files.upload()

## A2 — Load the data and show its size and column names

In [ ]:
train = pd.read_csv('/content/train.csv')
test  = pd.read_csv('/content/test.csv')
print('Train shape (rows, columns):', train.shape)
print('Test shape :', test.shape)
print('\nColumn names:', list(train.columns))
print('\nData types:\n', train.dtypes)

## A3 — First and last records, and basic info
`head()` shows the first 5 rows, `tail()` the last 5, `info()` summarises counts and types, `describe()` gives basic statistics.

In [ ]:
display(train.head())
display(train.tail())
train.info()
display(train.describe().T)

# B. Data Preprocessing
## B1 — Missing values and duplicates
Missing values and duplicate rows can break or bias a model, so we check them first.

In [ ]:
print('Missing values per column:\n', train.isna().sum()[train.isna().sum() > 0])
print('Total missing:', train.isna().sum().sum())
print('Duplicate rows:', train.duplicated().sum())

**Observation:** there are no missing values and no duplicates, so nothing needs removing.

## B2 — Numerical vs categorical variables
Columns with only 2 unique values (0/1) are binary flags (categorical). They are already encoded as numbers, so **no extra encoding is needed**.

In [ ]:
TARGET = 'price_range'
binary_cols  = [c for c in train.columns if train[c].nunique() == 2]
numeric_cols = [c for c in train.columns if c not in binary_cols + [TARGET]]
print('Binary/categorical:', binary_cols)
print('Numerical:', numeric_cols)
print('Target classes:', sorted(train[TARGET].unique()))

## B3 — Inconsistent values
A phone cannot have a screen width or pixel height of 0. We check for such values.

In [ ]:
print('px_height == 0 :', (train.px_height == 0).sum())
print('sc_w == 0       :', (train.sc_w == 0).sum())

**Fix:** replace those impossible zeros with the **median** of the valid values (the median is not affected by extreme values).

In [ ]:
for df in (train, test):
    for c in ['px_height', 'sc_w']:
        med = train.loc[train[c] > 0, c].median()
        df[c] = df[c].replace(0, med)
print('After fix -> px_height zeros:', (train.px_height == 0).sum(), '| sc_w zeros:', (train.sc_w == 0).sum())

## B4 — Split into training and validation sets, then scale
- `test.csv` has **no price_range**, so we cannot score on it. We split `train.csv` 80% / 20% to evaluate.
- **StandardScaler** gives every feature mean 0 and std 1. PCA is based on variance, so without scaling a big-number feature like `ram` would dominate.
- We fit the scaler on the training part only to avoid data leakage.

In [ ]:
X = train.drop(columns=TARGET); y = train[TARGET]
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
scaler = StandardScaler().fit(X_tr)
Xs_tr, Xs_val = scaler.transform(X_tr), scaler.transform(X_val)
Xs_all = StandardScaler().fit_transform(X)      # scaled full train data, used for exploration and plots
print('Train:', Xs_tr.shape, '| Validation:', Xs_val.shape)

# C. Exploratory Data Analysis (EDA)
## C1 — Target distribution (bar + pie)
Checks whether the classes are balanced.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
y.value_counts().sort_index().plot.bar(ax=ax[0], color='teal'); ax[0].set_title('Price range (bar)')
y.value_counts().sort_index().plot.pie(ax=ax[1], autopct='%1.0f%%'); ax[1].set_ylabel(''); ax[1].set_title('Price range (pie)')
plt.show()

**Observation:** all 4 classes have exactly 500 phones, so the data is balanced and accuracy is a fair metric.

## C2 — Histograms
Show the distribution (shape) of each numerical feature.

In [ ]:
train[numeric_cols].hist(figsize=(15, 10), bins=25, color='steelblue')
plt.tight_layout(); plt.show()

**Observation:** most features are roughly uniform (spread evenly), not bell-shaped. `fc` and `pc` are a bit skewed.

## C3 — Boxplots by price range
Show whether a feature changes as the price class changes, and show outliers.

In [ ]:
cols = ['ram','battery_power','px_height','px_width','int_memory','mobile_wt','talk_time','fc','pc','sc_h','sc_w','clock_speed']
fig, axs = plt.subplots(3, 4, figsize=(16, 10))
for a, c in zip(axs.ravel(), cols):
    sns.boxplot(data=train, x=TARGET, y=c, ax=a)
plt.tight_layout(); plt.show()

**Observation:** `ram` rises clearly with price. `battery_power`, `px_height` and `px_width` rise slightly. The others look the same across classes.

## C4 — Correlation heatmap
Shows which features move together (values from -1 to +1).

In [ ]:
plt.figure(figsize=(14, 11))
sns.heatmap(train.corr(), cmap='coolwarm', center=0, annot=True, fmt='.2f', annot_kws={'size': 6})
plt.title('Correlation heatmap'); plt.show()

**Observation:** only a few pairs are correlated: `fc`–`pc` (0.64), `four_g`–`three_g` (0.58), `px_height`–`px_width` (0.51), `sc_h`–`sc_w` (0.51). `ram` has 0.92 correlation with price. Everything else is near 0. **This matters for PCA** because PCA works by combining correlated features.

## C5 — Scatter plot and pair plot

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=train, x='ram', y='battery_power', hue=TARGET, palette='viridis', s=18)
plt.show()
sns.pairplot(train[['ram','battery_power','px_height','px_width',TARGET]], hue=TARGET, palette='viridis', plot_kws={'s': 8})
plt.show()

**Observation:** the colours separate along the `ram` axis (the price classes appear in horizontal bands), but not along the other features.

## C6 — Dendrogram (hierarchical clustering of features)
Groups features that behave similarly.

In [ ]:
Z = linkage(Xs_all.T, method='ward')
plt.figure(figsize=(11, 5))
dendrogram(Z, labels=X.columns.tolist(), leaf_rotation=90)
plt.title('Feature dendrogram (Ward)'); plt.show()

**Observation:** the correlated pairs (fc/pc, 3G/4G, px_height/px_width, sc_h/sc_w) join early, which agrees with the heatmap.

# D. Statistical Analysis
## D1 — Range, IQR, MAD, Variance, Std, Pearson correlation
- **Range** = max − min
- **IQR** = Q3 − Q1 (spread of the middle 50%)
- **MAD** = mean absolute deviation from the mean
- **Variance / Std** = spread around the mean
- **PCC** = linear correlation with the target `price_range`

In [ ]:
rows = []
for c in numeric_cols:
    s = train[c]
    rows.append(dict(feature=c, range=s.max()-s.min(), IQR=s.quantile(.75)-s.quantile(.25),
                     MAD=(s - s.mean()).abs().mean(), variance=s.var(), std=s.std(), PCC_with_price=s.corr(y)))
stat_tbl = pd.DataFrame(rows).round(3)
display(stat_tbl)

**Observation:** variances are very different (`ram` ≈ 1,176,000 vs `m_dep` ≈ 0.08). This is the reason we scaled. Only `ram` has a strong PCC with price (0.917).

## D2 — Outlier detection (IQR method and Z-score)
- IQR rule: outlier if below Q1 − 1.5·IQR or above Q3 + 1.5·IQR
- Z-score rule: outlier if |z| > 3

In [ ]:
out = []
for c in numeric_cols:
    q1, q3 = train[c].quantile([.25, .75]); i = q3 - q1
    n_iqr = ((train[c] < q1 - 1.5*i) | (train[c] > q3 + 1.5*i)).sum()
    n_z = (np.abs(stats.zscore(train[c])) > 3).sum()
    out.append((c, n_iqr, n_z))
display(pd.DataFrame(out, columns=['feature', 'IQR outliers', 'Z>3 outliers']))

**Observation:** outliers are rare (only in `fc`). These are real phones with high-resolution cameras, not errors, so we **keep** them.

# E. Model — PCA
## E1 — Fit PCA and look at explained variance
PCA creates new axes (**principal components**) ordered by how much variance they capture. We check how many are needed to keep 80%, 90% and 95% of the information.

In [ ]:
pca = PCA().fit(Xs_all)
evr = pca.explained_variance_ratio_
cum = np.cumsum(evr)
for t in (.80, .90, .95):
    print(f'Components needed for {int(t*100)}% variance:', int(np.argmax(cum >= t)) + 1)
print('Components with eigenvalue > 1 (Kaiser rule):', int((pca.explained_variance_ > 1).sum()))
print('Variance of first 5 PCs:', np.round(evr[:5], 3))

## E2 — Scree plot and cumulative variance plot

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(range(1, 21), evr, color='steelblue'); ax[0].plot(range(1, 21), evr, 'ro-')
ax[0].set(title='Scree plot', xlabel='Component', ylabel='Explained variance ratio')
ax[1].plot(range(1, 21), cum, 'go-')
ax[1].axhline(.95, ls='--', c='r'); ax[1].axhline(.90, ls='--', c='orange')
ax[1].set(title='Cumulative explained variance', xlabel='Number of components')
plt.tight_layout(); plt.show()

**Observation:** there is **no sharp elbow**. Each component explains only 3–8%, and the curve rises almost in a straight line. That means PCA cannot compress this data much.

## E3 — Loadings (what each component is made of)
A loading shows how much each original feature contributes to a component.

In [ ]:
load = pd.DataFrame(pca.components_[:5].T, index=X.columns, columns=[f'PC{i}' for i in range(1, 6)])
plt.figure(figsize=(7, 8))
sns.heatmap(load, cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('PCA loadings (first 5 PCs)'); plt.show()

**Observation:** PC1 is mostly `fc` + `pc` (cameras), PC2 is 3G/4G, PC3 is pixel size, PC4 is screen size. Each PC mainly captures one correlated pair. `ram` hardly appears here.

## E4 — Reduced-dimension visualization (2-D)
Project the 20 features down to 2 and colour by price class.

In [ ]:
P = PCA(n_components=2).fit_transform(Xs_all)
plt.figure(figsize=(7, 6))
sc = plt.scatter(P[:, 0], P[:, 1], c=y, cmap='viridis', s=10)
plt.colorbar(sc, label='price_range'); plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('2-D PCA projection')
plt.show()

**Observation:** the 4 colours are completely mixed. The first 2 components do **not** separate the price classes.

## E5 — Where does `ram` appear, and which components relate to price?

In [ ]:
S = pca.transform(Xs_all)
print('RAM loads most on PC', int(np.argmax(np.abs(pca.components_[:, list(X.columns).index('ram')]))) + 1)
corr_pc = [np.corrcoef(S[:, i], y)[0, 1] for i in range(20)]
display(pd.Series(np.round(corr_pc, 2), index=[f'PC{i}' for i in range(1, 21)]).to_frame('corr with price'))

**Observation:** `ram` (the key feature) lands in a later component (PC6) because PCA sorts by variance, not by usefulness for price. This is the main limitation.

# F. Prediction and Evaluation
## F1 — Baseline model: Logistic Regression on all 20 features
We use a simple classifier to test whether PCA keeps the important information. Metrics: accuracy, precision, recall, F1, ROC-AUC (macro-averaged over 4 classes).

In [ ]:
def evaluate(model, Xa, Xb, ya, yb):
    model.fit(Xa, ya); p = model.predict(Xb); pr = model.predict_proba(Xb)
    return dict(Accuracy=accuracy_score(yb, p), Precision=precision_score(yb, p, average='macro'),
                Recall=recall_score(yb, p, average='macro'), F1=f1_score(yb, p, average='macro'),
                ROC_AUC=roc_auc_score(yb, pr, multi_class='ovr')), p

res = {}
res['All 20 features'], p_full = evaluate(LogisticRegression(max_iter=2000), Xs_tr, Xs_val, y_tr, y_val)
display(pd.DataFrame(res).T.round(4))

## F2 — Same model on PCA-reduced data
We keep only k components (2, 5, 10, 15, 17) and compare with the baseline.

In [ ]:
for k in (2, 5, 10, 15, 17):
    pk = PCA(n_components=k).fit(Xs_tr)
    res[f'PCA {k} comps'], _ = evaluate(LogisticRegression(max_iter=2000), pk.transform(Xs_tr), pk.transform(Xs_val), y_tr, y_val)
comp = pd.DataFrame(res).T.round(4)
display(comp)

**Observation:** accuracy collapses with few components (2 → about 25%, which is random guessing) and only recovers at about 17. PCA does **not** preserve the information needed for price until almost all components are kept.

## F3 — Hyperparameter tuning (GridSearchCV)
We search the best number of components and the regularization strength `C` using 5-fold cross-validation on the training set.

In [ ]:
pipe = Pipeline([('pca', PCA()), ('lr', LogisticRegression(max_iter=2000))])
grid = {'pca__n_components': [2, 5, 8, 10, 12, 15, 17, 20], 'lr__C': [0.01, 0.1, 1, 10]}
gs = GridSearchCV(pipe, grid, cv=StratifiedKFold(5, shuffle=True, random_state=RS), scoring='accuracy').fit(Xs_tr, y_tr)
print('Best parameters:', gs.best_params_)
print('Best CV accuracy:', round(gs.best_score_, 4))
best = gs.best_estimator_
pb = best.predict(Xs_val)
print('Validation accuracy of tuned model:', round(accuracy_score(y_val, pb), 4))

## F4 — Confusion matrices
Rows are true classes and columns are predicted classes. A strong diagonal means good predictions.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_val, p_full)).plot(ax=ax[0], colorbar=False); ax[0].set_title('All 20 features')
ConfusionMatrixDisplay(confusion_matrix(y_val, pb)).plot(ax=ax[1], colorbar=False)
ax[1].set_title(f"Tuned PCA ({gs.best_params_['pca__n_components']} comps)")
plt.tight_layout(); plt.show()

**Observation:** both matrices are strongly diagonal; mistakes happen only between neighbouring classes (e.g. 1 vs 2).

# G. Training / Evaluation Curves
## G1 — Reconstruction error and accuracy vs number of components
- **Reconstruction error:** how much information is lost when we go back from k components to 20 features (lower = better).
- **Accuracy curve:** how performance changes as k increases.

In [ ]:
recon = []
for k in range(1, 21):
    m = PCA(n_components=k).fit(Xs_tr)
    recon.append(mean_squared_error(Xs_val, m.inverse_transform(m.transform(Xs_val))))
cvk = pd.DataFrame(gs.cv_results_)
cvk = cvk[cvk.param_lr__C == gs.best_params_['lr__C']]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(range(1, 21), recon, 'mo-'); ax[0].set(title='Reconstruction error vs components', xlabel='Components', ylabel='MSE')
ax[1].plot(cvk.param_pca__n_components, cvk.mean_test_score, 'bo-', label='CV accuracy')
ax[1].axhline(res['All 20 features']['Accuracy'], ls='--', c='r', label='All features')
ax[1].set(title='Accuracy vs PCA components', xlabel='Components'); ax[1].legend()
plt.tight_layout(); plt.show()

## G2 — Learning curve
Training vs validation accuracy as the amount of training data grows. If both curves are close and high, the model is not overfitting.

In [ ]:
sizes, trs, vas = learning_curve(best, Xs_all, y, cv=5, train_sizes=np.linspace(.1, 1, 8), scoring='accuracy', random_state=RS)
plt.figure(figsize=(7, 5))
plt.plot(sizes, trs.mean(1), 'o-', label='Training'); plt.plot(sizes, vas.mean(1), 'o-', label='Validation')
plt.xlabel('Training size'); plt.ylabel('Accuracy'); plt.title('Learning curve (PCA + Logistic Regression)'); plt.legend()
plt.show()

## G3 — Clustering on the PCA data (K-Means)
We test whether the PCA space naturally forms the 4 price groups. Measures: **WCSS/inertia** (elbow) and **silhouette score** (−1 to 1; higher is better).

In [ ]:
P10 = PCA(n_components=10).fit_transform(Xs_all)
inertia, sil = [], []
for k in range(2, 9):
    km = KMeans(k, n_init=10, random_state=RS).fit(P10)
    inertia.append(km.inertia_); sil.append(silhouette_score(P10, km.labels_))
km4 = KMeans(4, n_init=10, random_state=RS).fit(P10)
print('Silhouette score for k=4:', round(silhouette_score(P10, km4.labels_), 4))

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(range(2, 9), inertia, 'o-'); ax[0].set(title='Elbow (WCSS)', xlabel='k')
ax[1].plot(range(2, 9), sil, 'o-'); ax[1].set(title='Silhouette score', xlabel='k')
ax[2].scatter(P[:, 0], P[:, 1], c=km4.labels_, cmap='tab10', s=8); ax[2].set_title('K-Means (k=4) on PCA')
plt.tight_layout(); plt.show()

**Observation:** silhouette is very low (about 0.09) and there is no clear elbow. The data has no natural clusters, so the price groups are not found by unsupervised grouping.

## G4 — Predict the prices for `test.csv`
Train the final pipeline (scaler + PCA + Logistic Regression) on all labelled data, then predict the unlabelled test phones.

In [ ]:
final = Pipeline([('sc', StandardScaler()),
                  ('pca', PCA(n_components=gs.best_params_['pca__n_components'])),
                  ('lr', LogisticRegression(C=gs.best_params_['lr__C'], max_iter=2000))]).fit(X, y)
pred = pd.DataFrame({'id': test['id'], 'predicted_price_range': final.predict(test[X.columns])})
display(pred.head())
print(pred.predicted_price_range.value_counts())
pred.to_csv('/content/test_predictions.csv', index=False)
files.download('/content/test_predictions.csv')

# H. Interpretation — write these in your own words
Use your outputs above to answer. Hints are given after each question.

1. **What did the model learn?** (Features are mostly independent; price depends mainly on `ram`.)
2. **Which features were important?** (`ram`, then `battery_power`, `px_width`, `px_height`. In PCA the loadings are camera, 3G/4G, pixels and screen size.)
3. **How well did it perform?** (Quote your accuracy / F1 / ROC-AUC from F1–F3.)
4. **Patterns discovered?** (Four correlated pairs; classes balanced; `ram` rises with price.)
5. **Can PCA reduce dimensionality while preserving information? (main question)** (Only a little: about 17–18 of 20 components for 95% variance; with fewer, accuracy drops sharply.)
6. **Limitations?** (PCA is unsupervised; it ranks by variance, not by relevance to price; components are hard to interpret; linear method; no labels in test.csv.)
7. **Improvements?** (Use LDA or feature selection instead; try non-linear models; try t-SNE/UMAP.)
8. **Real-world decision?** (A company can price a new phone from its specs, or check whether a phone is overpriced.)